## BoardGameGeek — API

In [1]:
import time
import warnings

import pandas as pd
import requests
import xmltodict
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

BASE = "https://boardgamegeek.com/xmlapi2"
TOKEN = "7f5dd9f6-efef-41dd-8265-b88a15f97557"
HEADERS = {
    "User-Agent": "DataScienceClass/1.0",
    "Authorization": f"Bearer {TOKEN}",
}

In [2]:
def bgg_get(endpoint: str, params: dict = None, retries: int = 10) -> dict:
    if params is None:
        params = {}
    url = f"{BASE}/{endpoint}"
    for attempt in range(retries):
        r = requests.get(url, params=params, headers=HEADERS)
        if r.status_code == 200:
            return xmltodict.parse(r.text)
        elif r.status_code == 202:
            wait = 3 * (attempt + 1)
            print(f"  BGG queuing (attempt {attempt + 1}), retrying in {wait}s...")
            time.sleep(wait)
        elif r.status_code == 429:
            wait = 10 * (attempt + 1)
            print(f"  Rate limited (attempt {attempt + 1}), retrying in {wait}s...")
            time.sleep(wait)
        elif r.status_code == 401:
            raise Exception(
                "HTTP 401 — token missing or wrong.\n"
                "Get one at: https://boardgamegeek.com/applications"
            )
        else:
            raise Exception(f"HTTP {r.status_code} at {url}")
    raise Exception("Max retries exceeded.")


def get_primary_name(name_field):
    names = name_field if isinstance(name_field, list) else [name_field]
    return next(
        (n["@value"] for n in names if n.get("@type") == "primary"), "Unknown"
    )


def get_links(links, link_type):
    if isinstance(links, dict):
        links = [links]
    return ", ".join(v["@value"] for v in links if v.get("@type") == link_type)

In [3]:
# Currency snapshot: 2026/05/27
EXCHANGE_RATES_TO_USD = {
    "USD": 1.00,
    "EUR": 1.08,
    "GBP": 1.27,
    "CAD": 0.74,
    "AUD": 0.65,
    "CHF": 1.12,
    "SEK": 0.092,
    "NOK": 0.091,
    "DKK": 0.145,
    "JPY": 0.0067,
    "NZD": 0.60,
    "BRL": 0.19,
    "MXN": 0.058,
    "PLN": 0.25,
    "CZK": 0.044,
    "HUF": 0.0027,
    "RON": 0.22,
    "HRK": 0.14,
    "RUB": 0.011,
    "CNY": 0.138,
    "KRW": 0.00073,
    "SGD": 0.74,
    "HKD": 0.128,
    "ZAR": 0.054,
    "INR": 0.012,
}


def to_usd(price: float, currency: str) -> float | None:
    """Convert a price in any currency to USD using hardcoded rates."""
    rate = EXCHANGE_RATES_TO_USD.get(currency.upper())
    if rate is None:
        return None
    return round(price * rate, 2)

In [4]:
def fetch_game_details(ids: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fetch full metadata + marketplace listings + play log count
    for a list of BGG game IDs (max ~20 per /thing call).

    Returns
    -------
    games_df        : one row per game, includes num_logged
    marketplace_df  : one row per marketplace listing, price in USD
    """

    # Single /thing call with marketplace=1
    data = bgg_get("thing", {
        "id": ",".join(str(i) for i in ids),
        "stats": 1,
        "marketplace": 1,
    })
    items = data.get("items", {}).get("item", [])
    if isinstance(items, dict):
        items = [items]

    game_records = []
    market_records = []

    for item in items:
        game_id = int(item["@id"])
        name = get_primary_name(item.get("name", []))
        s = item.get("statistics", {}).get("ratings", {})
        links = item.get("link", [])
        if isinstance(links, dict):
            links = [links]

        # Game metadata record
        game_records.append({
            "id": game_id,
            "name": name,
            "year": item.get("yearpublished", {}).get("@value"),
            "min_players": item.get("minplayers", {}).get("@value"),
            "max_players": item.get("maxplayers", {}).get("@value"),
            "min_playtime": item.get("minplaytime", {}).get("@value"),
            "max_playtime": item.get("maxplaytime", {}).get("@value"),
            "min_age": item.get("minage", {}).get("@value"),
            "complexity": s.get("averageweight", {}).get("@value"),
            "avg_rating": s.get("average", {}).get("@value"),
            "bayes_rating": s.get("bayesaverage", {}).get("@value"),
            "num_ratings": s.get("usersrated", {}).get("@value"),
            "num_owned": s.get("owned", {}).get("@value"),
            "num_comments": s.get("numcomments", {}).get("@value"),
            "num_wanting": s.get("wanting", {}).get("@value"),
            "num_wishing": s.get("wishing", {}).get("@value"),
            "categories": get_links(links, "boardgamecategory"),
            "mechanics": get_links(links, "boardgamemechanic"),
            "designers": get_links(links, "boardgamedesigner"),
            "publishers": get_links(links, "boardgamepublisher"),
            "families": get_links(links, "boardgamefamily"),
            "num_logged": None,  # filled in below via /plays
        })

        # Marketplace listings
        listings = (
            item.get("marketplacelistings", {}).get("listing", [])
        )
        if isinstance(listings, dict):
            listings = [listings]

        for listing in listings:
            currency = listing.get("price", {}).get("@currency", "USD")
            price_orig = listing.get("price", {}).get("@value")
            try:
                price_orig = float(price_orig)
            except (TypeError, ValueError):
                price_orig = None

            market_records.append({
                "id": game_id,
                "name": name,
                "date": listing.get("listdate", {}).get("@value"),
                "price_orig": price_orig,
                "currency": currency,
                "price_usd": to_usd(price_orig, currency) if price_orig else None,
                "condition": listing.get("condition", {}).get("@value"),
            })

    # Build dataframes
    games_df = pd.DataFrame(game_records)

    num_cols = [
        "year", "min_players", "max_players", "min_playtime", "max_playtime",
        "min_age", "complexity", "avg_rating", "bayes_rating",
        "num_ratings", "num_owned", "num_comments", "num_wanting",
        "num_wishing",
    ]
    games_df[num_cols] = games_df[num_cols].apply(pd.to_numeric, errors="coerce")

    marketplace_df = pd.DataFrame(market_records)
    if not marketplace_df.empty:
        marketplace_df["date"] = pd.to_datetime(
            marketplace_df["date"], format="mixed", errors="coerce"
        ).dt.tz_localize(None)
        marketplace_df["year_month"] = marketplace_df["date"].dt.to_period("M")

    return games_df, marketplace_df

In [5]:
# `boardgames_ranks.csv` contains all game IDs.
# Use the IDs to fetch detailed BGG data.
ranks_df = pd.read_csv("boardgames_ranks.csv")
all_ids = ranks_df["id"].tolist()
print(f"Total games to fetch: {len(all_ids)}")

KeyboardInterrupt: 

In [ ]:
# API can only fetch 20 games at a time. Use for loop to fetch all.
# WARNING: This cell will take about 10 hours to run.

all_details = []
all_marketplace = []
failed_batches = []
batch_size = 20

batches = [all_ids[i:i + batch_size] for i in range(0, len(all_ids), batch_size)]

for batch_num, batch in enumerate(tqdm(batches, desc="Fetching BGG data"), start=1):
    try:
        games_batch, market_batch = fetch_game_details(batch)
        all_details.append(games_batch)
        if not market_batch.empty:
            all_marketplace.append(market_batch)
    except Exception as e:
        tqdm.write(f"  ✗ Batch {batch_num} failed: {e}")
        failed_batches.extend(batch)

    # Checkpoint every 100 batches
    if batch_num % 100 == 0:
        checkpoint = pd.concat(all_details, ignore_index=True)
        checkpoint.to_csv(f"checkpoint_{batch_num}.csv", index=False)
    if batch_num % 1000 == 0:
        tqdm.write(f"  💾 Checkpoint saved ({len(checkpoint)} games)")

    time.sleep(2)

# Final save
games_df = pd.concat(all_details, ignore_index=True)
marketplace_df = (
    pd.concat(all_marketplace, ignore_index=True)
    if all_marketplace
    else pd.DataFrame()
)

In [ ]:
# Save to CSV file
games_df.to_csv("data/bgg_details_20260527.csv", index=False)
marketplace_df.to_csv("data/bgg_marketplace_20260527.csv", index=False)

## Wikipedia

In [ ]:
import requests
import pandas as pd

WIKI_HEADERS = {"User-Agent": "Mozilla/5.0 (DataScienceClass/1.0)"}
WIKI_URL = (
    "https://en.wikipedia.org/wiki/"
    "List_of_Game_of_the_Year_awards_(board_games)"
)

r = requests.get(WIKI_URL, headers=WIKI_HEADERS)
r.raise_for_status()
all_tables = pd.read_html(r.text)

In [ ]:
print(f"Found {len(all_tables)} tables on the page")
for i, t in enumerate(all_tables):
    print(f"  Table {i:2d}: {t.shape} columns={t.columns.tolist()[:5]}")
    print(t.head())

In [ ]:
# Table's title on wikipedia - snapshot 2026/05/16
AWARD_TABLE_MAP = {
    0: ("American Tabletop Awards", "Early Gamers"),
    1: ("American Tabletop Awards", "Casual Games"),
    2: ("American Tabletop Awards", "Strategy Games"),
    3: ("American Tabletop Awards", "Complex Games"),
    4: ("As d'Or / Golden Ace", "General"),
    5: ("Board Game Quest Awards", "General"),
    6: ("Deutscher Spiele Preis", "General"),
    7: ("Diamond Climber Awards", "General"),
    8: ("Jogo do Ano", "General"),
    9: ("Games magazine", "General"),
    10: ("Golden Geek Award (2020 onwards)", "General"),
    11: ("Golden Geek Award (Till 2019)", "General"),
    12: ("Le Diamant d'Or", "General"),
    13: ("Mensa Select", "General"),
    14: ("Premio JdA", "General"),
    15: ("Spiel des Jahres", "General"),
    16: ("Spiel des Jahre", "Kennerspiel des Jahres"),
    17: ("Spiel des Jahre", "Kinderspiel des Jahres"),
}

In [ ]:
def clean_citations(series):
    """Remove Wikipedia citation brackets like [15], [n], [citation needed]."""
    return (
        series.astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.strip()
    )


WINNER_KEYWORDS = ["winner", "game", "title", "name"]

award_frames = []

for idx, (award_name, category) in AWARD_TABLE_MAP.items():
    if idx >= len(all_tables):
        print(f"  ⚠ Table {idx} not found — skipping {award_name} / {category}")
        continue

    t = all_tables[idx].copy()

    # Normalize column names to lowercase
    t.columns = [str(c).lower().strip() for c in t.columns]

    # Find year column
    year_col = next((c for c in t.columns if "year" in c), None)
    if not year_col:
        print(f"  ⚠ Table {idx} ({award_name}): no year column found — skipping")
        continue

    # Find winner/name column
    name_col = next(
        (c for c in t.columns if any(kw in c for kw in WINNER_KEYWORDS)), None
    )

    # ── ROUTE SELECTION ─────────────────────────────────────
    # Rule 1: tuple[1] != "General" → use hardcoded category
    # Rule 2: tuple[1] == "General" AND df has 'category' col → use that column
    # Rule 3: tuple[1] == "General" AND no winner col found → melt()
    # Fallback: none of the above → skip with warning

    if category != "General":
        # RULE 1
        if not name_col:
            print(f"  ⚠ [FALLBACK] Table {idx} ({award_name} / {category}): "
                  f"no winner column — columns={t.columns.tolist()}")
            continue
        result = t[[year_col, name_col]].copy()
        result.columns = ["year", "game_name"]
        result["category"] = category
        path = "Rule 1 (hardcoded category)"

    elif "category" in t.columns and name_col:
        # RULE 2
        result = t[[year_col, "category", name_col]].copy()
        result.columns = ["year", "category", "game_name"]
        path = "Rule 2 (category from column)"

    elif not name_col:
        # RULE 3: melt
        value_cols = [c for c in t.columns if c != year_col]
        result = t.melt(
            id_vars=[year_col],
            value_vars=value_cols,
            var_name="category",
            value_name="game_name"
        ).rename(columns={year_col: "year"})
        path = f"Rule 3 (melt — cols: {value_cols})"

    else:
        # FALLBACK
        print(f"  ⚠ [FALLBACK] Table {idx} ({award_name}): "
              f"fell through all rules — columns={t.columns.tolist()}")
        result = t[[year_col, name_col]].copy()
        result.columns = ["year", "game_name"]
        result["category"] = "General"
        path = "Fallback (General)"

    # CLEAN & FINALIZE
    result["year"] = clean_citations(result["year"])
    result["year"] = pd.to_numeric(result["year"], errors="coerce")
    result["game_name"] = clean_citations(result["game_name"])
    result["category"] = clean_citations(result["category"])
    result["award"] = award_name

    result = result.dropna(subset=["year", "game_name"])
    result = result[result["game_name"] != ""]
    result["year"] = result["year"].astype(int)

    # Reorder columns consistently
    result = result[["year", "award", "category", "game_name"]]

    award_frames.append(result)
    print(f"  ✓ Table {idx:2d} [{path}]: {award_name} / {category} — {len(result)} rows")

# Combine all award tables
awards_df = pd.concat(award_frames, ignore_index=True)
awards_df["game_name"] = awards_df["game_name"].str.strip()

In [ ]:
# Save to CSV file
awards_df.to_csv("data/awards_dataset_20260516.csv")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b3536b60-e481-4f67-b46c-1c4de87b554b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>